### Notebook to generate the optimal combination of cmems of OSCAR. 

- Gives best case if we are able to alway pick which model gives the best forecast. 
- Gives dataset in the format of all othe other forecasts

In [16]:
import pandas as pd 
import numpy as np 
from functions.output_functions import haversine_df

In [17]:
ds = pd.read_csv(r"saved_output\cmems_dynamical_2022_2025_wind.csv") #combined_cmems2024.csv
ds1 = pd.read_csv(r"saved_output\OSCAR_2022_2025_wind.csv")

In [19]:
ds['error_km'] = haversine_df(ds, "lat_true", "lon_true", "lat_forcast", "lon_forcast")
ds1['error_km'] = haversine_df(ds1, "lat_true", "lon_true", "lat_forcast", "lon_forcast")

def add_inital_time(ds:pd.DataFrame):
    """adds intial time of forecast (time - leadtime) and changes collumns to pd.DateTime objects"""
    ds["Time"] = pd.to_datetime(ds["Time"])
    ds["initial_time"] =  ds["Time"] - pd.to_timedelta(ds["leadtime"], unit = "hours")
    ds["initial_time"] = ds["initial_time"].dt.round(freq = "min")
    return ds
ds = add_inital_time(ds)
ds1 = add_inital_time(ds1)
bins = np.linspace(0,8*24,2*24+1)
ds1["lead_bins"] = pd.cut(ds1["leadtime"], bins, right = True)
ds ["lead_bins"] = pd.cut(ds["leadtime"], bins, right = True)
dsi = ds.query("leadtime == 0").reset_index(drop = True)
ds1i = ds1.query("leadtime == 0").reset_index(drop = True)
print(len(dsi), len(ds1i)) ## same amount of inital forecasts

32041 33346


In [20]:
##merge on BuoyID and inital_time add OSCAR Forcast times to cmems forecasts

# Remove duplicates, keeping first occurrence
ds_clean = ds.drop_duplicates(subset=['BuoyID', 'initial_time', 'leadtime'], keep='first')
ds1_clean = ds1.drop_duplicates(subset=['BuoyID', 'initial_time', 'leadtime'], keep='first')

merge = pd.merge(ds_clean, ds1_clean, 
                 left_on=['BuoyID', 'leadtime', 'initial_time'],
                 right_on=['BuoyID', 'leadtime', 'initial_time'], 
                 how='outer',
                 suffixes=('_cmems', '_OSCAR'))
# merge= pd.merge(ds, ds1, 
#                      left_on=['BuoyID', 'leadtime', 'initial_time'],
#                      right_on=['BuoyID', 'leadtime', 'initial_time'], 
#                      how='outer',
#                      suffixes=('_cmems', '_OSCAR'))

print(f"max merging error(lat {(merge.lat_true_cmems - merge.lat_true_OSCAR).max()}")
print(f"Amount of intial forecasts {merge.query("leadtime == 0").shape[0]}")


max merging error(lat 0.0
Amount of intial forecasts 32995


In [21]:
# Define binlist and target_time outside the function (once, not per group)
binlist = merge.lead_bins_cmems.unique()
binlist = binlist.sort_values()
target_time = binlist[17]  # hour 68-72

def better_forecast(group): 
    maxcmemsbin = group.lead_bins_cmems.max() 
    maxoscarbin = group.lead_bins_OSCAR.max() 
    maxcmems = group.lead_bins_cmems.max() >= target_time if pd.notna(maxcmemsbin) else False
    maxoscar = group.lead_bins_OSCAR.max() >= target_time if pd.notna(maxoscarbin) else False
    
    if maxcmems and maxoscar:  # both forecasts longer than target time
        row = group.query("lead_bins_cmems == @target_time").reset_index(drop=True)
        if len(row) > 0:
            if row.iloc[0]["error_km_cmems"] < row.iloc[0]["error_km_OSCAR"]: 
                best = "cmems"
            else: 
                best = "OSCAR"
        else:
            # Fallback: pick the one with lower error
            best = "cmems" if group["error_km_cmems"].min() < group["error_km_OSCAR"].min() else "OSCAR"
    elif maxcmems: 
        best = "cmems"
    elif maxoscar: 
        best = "OSCAR"
    else: 
        best = "cmems"
    
    group["lat_forcast_opt"] = group[f"lat_forcast_{best}"]
    group["lon_forcast_opt"] = group[f"lon_forcast_{best}"]
    group["model"] = best
    return group

forecast = merge.groupby(["BuoyID", "initial_time"], observed = True).apply(better_forecast, include_groups=False).reset_index(level=['BuoyID', 'initial_time']).reset_index(drop = True)


In [22]:
forecast_simple = forecast.copy()
# Time_cmems is NaN for rows that only exist in the OSCAR dataset (outer merge miss on cmems side)
# Time_OSCAR is NaN for rows that only exist in cmems. Coalesce both — they are the same timestamp.
forecast_simple["Time"] = forecast["Time_cmems"].fillna(forecast["Time_OSCAR"])
# Same for lat/lon true columns
forecast_simple["lat_true"] = forecast["lat_true_cmems"].fillna(forecast["lat_true_OSCAR"])
forecast_simple["lon_true"] = forecast["lon_true_cmems"].fillna(forecast["lon_true_OSCAR"])

forecast_simple = forecast_simple.loc[:, ["BuoyID", "Time", "leadtime", "initial_time",
                                          "lat_true", "lon_true", "lat_forcast_opt", 
                                          "lon_forcast_opt", "model"]]
forecast_simple = forecast_simple.rename(columns={"lat_forcast_opt": "lat_forcast",
                                                  "lon_forcast_opt": "lon_forcast"})
print(f"NaN Times remaining: {forecast_simple.Time.isna().sum()}")


NaN Times remaining: 0


In [23]:
forecast_simple.to_csv(r"saved_output\optimal_cmems_OSCAR_2022_2024.csv")